# 🇻🇳 ViMind: Quy Trình Huấn Luyện Tự Động Từ Đầu (Pre-training & SFT)
Notebook này được tối ưu hóa cho Kaggle GPU (Tesla T4 / P100 16GB VRAM):
1. **Tự động clone mã nguồn từ GitHub.**
2. **Tự động tải dữ liệu tiếng Việt** (Wikipedia 405k bài viết & SFT 52k hội thoại) trong 2 phút.
3. **Tiền huấn luyện (Pre-training):** Huấn luyện mô hình ViMind 26M học ngữ pháp tiếng Việt.
4. **Tinh chỉnh chỉ dẫn (SFT):** Huấn luyện mô hình trở thành Trợ lý ảo đối thoại.

In [ ]:
# 1. Clone toàn bộ mã nguồn ViMind từ GitHub và di chuyển vào thư mục làm việc
!rm -rf /kaggle/working/vimind
!git clone https://github.com/WuKong0601/ViMind.git /kaggle/working/vimind
%cd /kaggle/working/vimind


In [ ]:
# 2. Cài đặt các thư viện cần thiết
!pip install -r requirements.txt


In [ ]:
# 3. Kiểm tra thông số GPU (Tesla T4 hoặc P100 16GB VRAM)
!nvidia-smi


In [ ]:
# 4. Tự động tải dữ liệu Pretrain và SFT qua mạng cáp quang tốc độ cao (khoảng 2 phút)
!python data_pipeline/download_pretrain.py
!python data_pipeline/download_sft.py


In [ ]:
# 5. Chạy Dry-Run kiểm tra GPU & kiểm tra cấu trúc mô hình
!python trainer/test_dry_run.py
!python trainer/test_sft_dry_run.py


In [ ]:
# 6. [GIAI ĐOẠN PRE-TRAINING] Bắt đầu huấn luyện mô hình nền trên toàn bộ dữ liệu tiếng Việt
!python trainer/pretrain.py \
    --data_path dataset/pretrain_vi.jsonl \
    --tokenizer_dir model \
    --save_dir out \
    --save_weight vimind_26m \
    --batch_size 32 \
    --accumulation_steps 4 \
    --epochs 1 \
    --dtype float16 \
    --log_interval 50 \
    --save_interval 1000


In [ ]:
# 7. [GIAI ĐOẠN SFT] Tinh chỉnh chỉ dẫn từ trọng số Pre-training để biến thành Chatbot
!python trainer/train_sft.py \
    --data_path dataset/sft_vi.jsonl \
    --tokenizer_dir model \
    --from_pretrained out/vimind_26m_final \
    --save_dir out/sft \
    --save_weight vimind_sft \
    --batch_size 16 \
    --accumulation_steps 4 \
    --epochs 2 \
    --learning_rate 1e-4 \
    --dtype float16 \
    --log_interval 25 \
    --save_interval 500


In [ ]:
# 8. [KIỂM THỬ THÀNH PHẨM] Trò chuyện thử với mô hình ViMind vừa được huấn luyện xong
import torch
from transformers import AutoTokenizer
from model.model import ViMindForCausalLM

model_path = "out/sft/vimind_sft_final"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = ViMindForCausalLM.from_pretrained(model_path).cuda()
model.eval()

test_prompts = [
    "Xin chào, bạn là ai và bạn có thể giúp gì cho tôi?",
    "Thủ đô của Việt Nam là gì?",
    "Hãy nêu 3 lợi ích của việc tập thể dục mỗi ngày."
]

print("=" * 60)
print("🎉 KẾT QUẢ TRẢ LỜI CỦA VIMIND THÀNH PHẨM:")
print("=" * 60)
for p in test_prompts:
    formatted = tokenizer.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").input_ids.cuda()
    with torch.no_grad():
        outputs = model.generate(inputs, max_new_tokens=150, temperature=0.7, eos_token_id=tokenizer.eos_token_id)
    ans = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"👤 Người dùng: {p}")
    print(f"🤖 ViMind: {ans}")
    print("-" * 60)
